In [5]:
import pandas as pd
pd.set_option("display.max_columns", None)

In [6]:
import json
import os
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from supabase import create_client

API_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(API_DIR / ".env")

SP500_TICKERS_PATH = API_DIR / "data" / "sp500_tickers.json"
SP500_INDEX_TICKER = "^GSPC"
RISK_FREE_TICKER = "^IRX"

with open(SP500_TICKERS_PATH, encoding="utf-8") as f:
    SP500_TICKERS: list[str] = json.load(f)

supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])

# daily_close_prices holds stocks + index; risk-free rate is fetched separately below.
close_tickers = [*SP500_TICKERS, SP500_INDEX_TICKER]

close_rows = []
PAGE_SIZE = 1000
for i in range(0, len(close_tickers), 50):
    chunk = close_tickers[i : i + 50]
    start = 0
    while True:
        page = (
            supabase.table("daily_close_prices")
            .select("date, ticker, close")
            .in_("ticker", chunk)
            .order("date")
            .order("ticker")
            .range(start, start + PAGE_SIZE - 1)
            .execute()
        ).data
        close_rows.extend(page)
        if len(page) < PAGE_SIZE:
            break
        start += PAGE_SIZE

close_df = pd.DataFrame(close_rows)
close_df["date"] = pd.to_datetime(close_df["date"])
close_df = close_df.drop_duplicates(subset=["date", "ticker"])

# Wide table: rows = date, columns = ticker (SP500 tickers + ^GSPC), values = close price
prices = close_df.pivot(index="date", columns="ticker", values="close").sort_index()

# Log-normal daily returns for stocks + index (not computed for the risk-free rate)
log_returns = np.log(prices / prices.shift(1)).dropna(how="all")

# Risk-free rate (annualized, from treasury bill data) — not converted to log returns
rf_rows = (
    supabase.table("daily_close_prices")
    .select("date, close")
    .eq("ticker", RISK_FREE_TICKER)
    .order("date")
    .execute()
).data
risk_free = pd.DataFrame(rf_rows)
risk_free["date"] = pd.to_datetime(risk_free["date"])
risk_free = risk_free.drop_duplicates(subset=["date"]).set_index("date")["close"]
risk_free = risk_free.rename("risk_free_rate").sort_index()

print(prices.shape, log_returns.shape, risk_free.shape)
prices.tail()


(251, 504) (250, 504) (252,)


ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,ADSK,AEE,AEP,AES,AFL,AIG,AIZ,AJG,AKAM,ALB,ALGN,ALL,ALLE,AMAT,AMCR,AMD,AME,AMGN,AMP,AMT,AMZN,ANET,AON,AOS,APA,APD,APH,APO,APP,APTV,ARE,ARES,ATO,AVB,AVGO,AVY,AWK,AXON,AXP,AZO,BA,BAC,BALL,BAX,BBY,BDX,BEN,BF-B,BG,BIIB,BKNG,BKR,BLDR,BLK,BMY,BNY,BR,BRK-B,BRO,BSX,BX,BXP,C,CAH,CARR,CASY,CAT,CB,CBOE,CBRE,CCI,CCL,CDNS,CDW,CEG,CF,CFG,CHD,CHRW,CHTR,CI,CIEN,CINF,CL,CLX,CMCSA,CME,CMG,CMI,CMS,CNC,CNP,COF,COHR,COIN,COO,COP,COR,COST,CPAY,CPRT,CPT,CRH,CRL,CRM,CRWD,CSCO,CSGP,CSX,CTAS,CTSH,CTVA,CVNA,CVS,CVX,D,DAL,DASH,DD,DDOG,DE,DECK,DELL,DG,DGX,DHI,DHR,DIS,DLR,DLTR,DOC,DOV,DOW,DPZ,DRI,DTE,DUK,DVA,DVN,DXCM,EBAY,ECHO,ECL,ED,EFX,EG,EIX,EL,ELV,EME,EMR,EOG,EQIX,EQR,EQT,ERIE,ES,ESS,ETN,ETR,EVRG,EW,EXC,EXE,EXPD,EXPE,EXR,F,FANG,FAST,FCX,FDS,FDX,FDXF,FE,FERG,FFIV,FICO,FIS,FISV,FITB,FIX,FLEX,FOX,FOXA,FRT,FSLR,FTNT,FTV,GD,GDDY,GE,GEHC,GEN,GEV,GILD,GIS,GL,GLW,GM,GNRC,GOOG,GOOGL,GPC,GPN,GRMN,GS,GWW,HAL,HAS,HBAN,HCA,HD,HIG,HII,HLT,HON,HONA,HOOD,HPE,HPQ,HRL,HSIC,HST,HSY,HUBB,HUM,HWM,IBKR,IBM,ICE,IDXX,IEX,IFF,INCY,INTC,INTU,INVH,IP,IQV,IR,IRM,ISRG,IT,ITW,IVZ,J,JBHT,JBL,JCI,JKHY,JNJ,JPM,KDP,KEY,KEYS,KHC,KIM,KKR,KLAC,KMB,KMI,KO,KR,KVUE,L,LDOS,LEN,LH,LHX,LII,LIN,LITE,LLY,LMT,LNT,LOW,LRCX,LULU,LUV,LVS,LYB,LYV,MA,MAA,MAR,MAS,MCD,MCHP,MCK,MCO,MDLZ,MDT,MET,META,MGM,MKC,MLM,MMM,MNST,MO,MOS,MPC,MPWR,MRK,MRNA,MRSH,MRVL,MS,MSCI,MSFT,MSI,MTB,MTD,MU,NCLH,NDAQ,NDSN,NEE,NEM,NFLX,NI,NKE,NOC,NOW,NRG,NSC,NTAP,NTRS,NUE,NVDA,NVR,NWS,NWSA,NXPI,O,ODFL,OKE,OMC,ON,ORCL,ORLY,OTIS,OXY,PANW,PAYX,PCAR,PCG,PEG,PEP,PFE,PFG,PG,PGR,PH,PHM,PKG,PLD,PLTR,PM,PNC,PNR,PNW,PODD,PPG,PPL,PRU,PSA,PSKY,PSX,PTC,PWR,PYPL,Q,QCOM,RCL,REG,REGN,RF,RJF,RL,RMD,ROK,ROL,ROP,ROST,RSG,RTX,RVTY,SBAC,SBUX,SCHW,SHW,SJM,SLB,SMCI,SNA,SNDK,SNPS,SO,SOLV,SPG,SPGI,SRE,STE,STLD,STT,STX,STZ,SW,SWK,SWKS,SYF,SYK,SYY,T,TAP,TDG,TDY,TECH,TEL,TER,TFC,TGT,TJX,TKO,TMO,TMUS,TPL,TPR,TRGP,TRMB,TROW,TRV,TSCO,TSLA,TSN,TT,TTD,TTWO,TXN,TXT,TYL,UAL,UBER,UDR,UHS,ULTA,UNH,UNP,UPS,URI,USB,V,VEEV,VICI,VLO,VLTO,VMC,VRSK,VRSN,VRT,VRTX,VST,VTR,VTRS,VZ,WAB,WAT,WBD,WDAY,WDC,WEC,WELL,WFC,WM,WMB,WMT,WRB,WSM,WST,WTW,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS,^GSPC
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-10,148.72,308.26,247.97,184.700,108.62,98.25,178.25,272.96,383.93,80.49,273.68,256.07,107.00,122.84,14.73,121.94,77.61,287.22,252.43,117.65,131.21,176.04,269.67,165.53,522.12,47.11,469.56,252.88,417.20,557.53,169.11,278.09,191.52,356.97,62.41,41.02,308.18,167.70,132.02,339.00,49.81,48.35,137.66,167.92,183.84,422.40,178.35,135.45,596.33,338.86,3067.9900,232.79,63.86,62.58,27.78,82.43,179.63,33.52,28.19,112.65,207.13,212.88,64.07,71.98,1131.40,64.84,159.43,173.40,529.42,71.60,50.46,141.66,67.90,135.22,237.18,63.44,854.46,837.58,348.30,285.02,143.94,73.61,27.75,331.91,136.43,270.43,121.48,72.67,103.30,148.29,153.29,278.40,387.70,173.86,93.15,107.70,25.20,263.65,32.14,634.82,69.06,66.57,40.03,218.97,325.15,148.68,76.32,123.03,325.00,952.75,400.88,29.59,109.08,100.60,277.00,197.51,225.16,122.57,31.48,50.12,202.71,58.31,76.47,74.16,95.67,194.91,67.15,89.21,209.86,141.55,260.78,609.28,97.44,457.88,122.42,237.63,146.66,209.37,103.18,191.30,129.37,20.90,210.10,30.58,347.85,217.97,136.78,121.19,183.69,45.36,87.65,107.71,89.15,284.40,106.30,181.04,369.90,67.83,87.19,397.70,809.33,158.71,142.22,1043.38,65.83,54.08,252.35,71.17,283.21,444.96,105.65,82.09,92.03,44.50,97.56,179.00,314.01,146.66,14.00,198.97,52.15,70.51,284.12,325.08,141.35,46.76,263.78,409.99,1049.74,42.51,52.22,57.13,1674.8400,119.70,56.33,63.42,117.13,239.33,164.21,61.75,395.97,91.

In [76]:
import numpy as np
from pypfopt import expected_returns
from pypfopt import risk_models
from pypfopt import EfficientFrontier

# -----------------------------
# 1. Expected returns
# -----------------------------
mu = expected_returns.mean_historical_return(prices)

# Cap extreme historical estimates
mu_capped = mu.clip(lower=-0.50, upper=0.50)

# -----------------------------
# 2. Covariance matrix
# -----------------------------
S = risk_models.CovarianceShrinkage(prices).ledoit_wolf()

# -----------------------------
# 3. Risk-free rate
# -----------------------------
rf = risk_free[1].mean() / 100

# -----------------------------
# 4. Portfolio optimisation
# -----------------------------
ef = EfficientFrontier(
    mu_capped,
    S,
    weight_bounds=(0, 1),
    solver="CLARABEL"
)

# Maximum 3% in any individual stock
ef.add_constraint(lambda w: w <= 0.03)

# Maximum Sharpe / tangency portfolio
weights = ef.max_sharpe(risk_free_rate=rf)

weights = ef.clean_weights()

# -----------------------------
# 5. Portfolio statistics
# -----------------------------
ret, volatility, sharpe = ef.portfolio_performance(
    risk_free_rate=rf,
    verbose=True
)

C:\Users\yajat\projects\value-investing-dash\api\.venv\lib\site-packages\pypfopt\expected_returns.py:32: UserWarning: Some returns are NaN. Please check your price data.
  warnings.warn(
C:\Users\yajat\AppData\Local\Temp\ipykernel_20492\1390584183.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  rf = risk_free[1].mean() / 100


Expected annual return: 43.8%
Annual volatility: 7.7%
Sharpe Ratio: 5.14


Calculating the expected returns of a portfolio given above.

In [104]:
weights_df = pd.DataFrame.from_dict(
    weights,
    orient='index',
    columns=['Weight']
)

weights_df.index.name = 'Ticker'
sigmaP = weights_df['Weight'].values.transpose() @S.values @ weights_df['Weight'].values
sigmaM = S['^GSPC']['^GSPC']
ERp = (sigmaP/sigmaM)*(mu['^GSPC'] - rf) + rf
print(f"Max Sharpe Portfolio Expected Returns: {ERp:.2%} vs actual return: {ret:.2%}")

Max Sharpe Portfolio Expected Returns: 7.64% vs actual return: 43.84%


Creating a efficient portfolio envelope given any envolope portfolio can be done by using the markowitz theorum. remmember to adhere to the same constraints.